# Regularisation — L2 and Dropout

> Based on Stanford CS230 — C2M1, Andrew Ng / DeepLearning.AI

Regularisation reduces overfitting (high variance) by constraining the model. The two most widely used techniques are **L2 weight decay** and **dropout**.

## Learning Objectives

1. Add L2 regularisation to the cost function and its gradients
2. Implement inverted dropout and understand why it preserves expected activations
3. Visualise how regularisation strength affects the decision boundary
4. Explain *why* L2 drives weights small and *why* dropout acts as ensemble averaging

## L2 Regularisation (Weight Decay)

The regularised cost adds a penalty proportional to the squared norm of all weight matrices:

$$J_{\text{reg}}(W, b) = J(W, b) + \frac{\lambda}{2m} \sum_{l=1}^{L} \|W^{[l]}\|_F^2$$

where $\|\cdot\|_F$ is the Frobenius norm. The gradient update becomes:

$$dW^{[l]} \leftarrow dW^{[l]} + \frac{\lambda}{m} W^{[l]}$$

This is equivalent to multiplying weights by $(1 - \alpha \lambda / m)$ each step — hence **weight decay**.

## Dropout — Inverted Dropout

At each training iteration, each unit is kept with probability $p$ (keep-prob):

$$d^{[l]} \sim \text{Bernoulli}(p), \quad A^{[l]} \leftarrow A^{[l]} \odot d^{[l]} / p$$

Dividing by $p$ (inverted dropout) ensures $\mathbb{E}[A^{[l]}]$ is unchanged, so **no scaling is needed at test time**.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams.update({
    'figure.facecolor': '#f5f5f7', 'axes.facecolor': '#ffffff',
    'axes.edgecolor': '#c8ccd4', 'axes.labelcolor': '#1a1d27',
    'axes.titlecolor': '#1a1d27', 'xtick.color': '#2a2e3a',
    'ytick.color': '#2a2e3a', 'grid.color': '#e0e3ea',
    'grid.linestyle': '--', 'grid.alpha': 0.5,
    'text.color': '#1a1d27', 'font.family': 'DejaVu Sans',
    'axes.titlesize': 13, 'axes.labelsize': 11,
    'legend.facecolor': '#ffffff', 'legend.edgecolor': '#c8ccd4',
    'figure.dpi': 110,
})
P = ['#5b9bd5', '#e05c5c', '#f4b942', '#7ecba1', '#56b6c2', '#c678dd']

# ---- helpers ----
def sigmoid(z): return 1 / (1 + np.exp(-z))
def relu(z):    return np.maximum(0, z)

def init_he(dims, seed=1):
    np.random.seed(seed)
    p = {}
    for l in range(1, len(dims)):
        p[f'W{l}'] = np.random.randn(dims[l], dims[l-1]) * np.sqrt(2/dims[l-1])
        p[f'b{l}'] = np.zeros((dims[l], 1))
    return p

def forward(X, params, L, keep_prob=1.0, training=True):
    caches, A = [], X
    masks = {}
    for l in range(1, L+1):
        W, b = params[f'W{l}'], params[f'b{l}']
        Z = W @ A + b
        A_prev = A
        A = relu(Z) if l < L else sigmoid(Z)
        if training and l < L and keep_prob < 1.0:
            mask = (np.random.rand(*A.shape) < keep_prob) / keep_prob
            A = A * mask
            masks[l] = mask
        caches.append((A_prev, W, b, Z, A))
    return A, caches, masks

def cost(AL, Y, params, L, lam):
    m = Y.shape[1]
    ce = -np.mean(Y*np.log(AL+1e-8) + (1-Y)*np.log(1-AL+1e-8))
    reg = (lam/(2*m)) * sum(np.sum(params[f'W{l}']**2) for l in range(1, L+1))
    return ce + reg

def backward(caches, masks, params, Y, L, lam, keep_prob=1.0):
    grads = {}
    m = Y.shape[1]
    _, _, _, _, AL = caches[-1]
    dA = -(Y/(AL+1e-8) - (1-Y)/(1-AL+1e-8))
    for l in reversed(range(1, L+1)):
        A_prev, W, b, Z, A = caches[l-1]
        if l == L:
            dZ = dA * A * (1-A)
        else:
            dZ = dA * (Z > 0).astype(float)
        grads[f'dW{l}'] = (dZ @ A_prev.T) / m + (lam/m) * W
        grads[f'db{l}'] = np.mean(dZ, axis=1, keepdims=True)
        dA = W.T @ dZ
        if l-1 in masks:
            dA = dA * masks[l-1]
    return grads

def train(X, Y, dims, lam=0.0, keep_prob=1.0, lr=0.3, n=3000, seed=1):
    L = len(dims)-1
    params = init_he(dims, seed=seed)
    costs = []
    for _ in range(n):
        AL, caches, masks = forward(X, params, L, keep_prob, training=True)
        costs.append(cost(AL, Y, params, L, lam))
        grads = backward(caches, masks, params, Y, L, lam, keep_prob)
        for l in range(1, L+1):
            params[f'W{l}'] -= lr * grads[f'dW{l}']
            params[f'b{l}'] -= lr * grads[f'db{l}']
    return params, costs

# ---- Dataset: noisy moon-like data ----
np.random.seed(42)
m = 300
t1 = np.linspace(0, np.pi, m//2)
X = np.vstack([
    np.c_[np.cos(t1) + np.random.randn(m//2)*0.2, np.sin(t1) + np.random.randn(m//2)*0.2],
    np.c_[1 - np.cos(t1) + np.random.randn(m//2)*0.2, 0.5 - np.sin(t1) + np.random.randn(m//2)*0.2]
]).T
Y = np.array([1]*(m//2) + [0]*(m//2)).reshape(1,-1)
dims = [2, 20, 10, 1]

configs = [
    ('No regularisation (λ=0)',    0.0,  1.0,  P[1]),
    ('L2  λ=0.3',                  0.3,  1.0,  P[0]),
    ('L2  λ=1.0',                  1.0,  1.0,  P[2]),
    ('Dropout  p=0.7',             0.0,  0.7,  P[3]),
]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
fig.suptitle('Effect of Regularisation on Decision Boundary', fontsize=13, fontweight='bold')

h = 0.04
xx, yy = np.meshgrid(np.arange(-0.5, 1.7, h), np.arange(-0.4, 1.5, h))
grid = np.c_[xx.ravel(), yy.ravel()].T

for col, (label, lam, kp, color) in enumerate(configs):
    params, costs = train(X, Y, dims, lam=lam, keep_prob=kp)
    L = len(dims) - 1

    # cost curve
    axes[0][col].plot(costs, color=color, lw=2)
    axes[0][col].set_title(label, fontsize=9)
    axes[0][col].set_xlabel('Iteration'); axes[0][col].set_ylabel('Cost' if col==0 else '')
    axes[0][col].grid(True)

    # decision boundary (test-time: no dropout)
    probs, _, _ = forward(grid, params, L, keep_prob=1.0, training=False)
    probs = probs.reshape(xx.shape)
    axes[1][col].contourf(xx, yy, probs, levels=50, cmap='RdBu_r', alpha=0.5)
    axes[1][col].scatter(X[0, Y[0]==1], X[1, Y[0]==1], color=P[0], s=15, alpha=0.7)
    axes[1][col].scatter(X[0, Y[0]==0], X[1, Y[0]==0], color=P[1], s=15, alpha=0.7)
    axes[1][col].set_title(f'Boundary — {label}', fontsize=9)
    axes[1][col].grid(True)

plt.tight_layout()
plt.show()

# Weight norm comparison
for label, lam, kp, _ in configs:
    params, _ = train(X, Y, dims, lam=lam, keep_prob=kp)
    L = len(dims)-1
    w_norm = np.sqrt(sum(np.sum(params[f'W{l}']**2) for l in range(1, L+1)))
    print(f"{label:35s}  ||W||_F = {w_norm:.3f}")


## Dropout — How It Works

<svg xmlns="http://www.w3.org/2000/svg" viewBox="0 0 660 200" width="660" height="200" style="font-family:DejaVu Sans,Arial,sans-serif;background:#f5f5f7;border-radius:8px">
  <!-- Title labels -->
  <text x="165" y="18" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">Training (p = 0.5)</text>
  <text x="495" y="18" text-anchor="middle" fill="#333" font-size="11" font-weight="bold">Test time (no dropout)</text>
  <line x1="330" y1="10" x2="330" y2="195" stroke="#e0e3ea" stroke-width="1" stroke-dasharray="4,3"/>

  <!-- Training side: connections -->
  <g stroke="#dde0e8" stroke-width="0.9">
    <line x1="60" y1="50" x2="200" y2="50"/><line x1="60" y1="50" x2="200" y2="100"/>
    <line x1="60" y1="100" x2="200" y2="50"/><line x1="60" y1="100" x2="200" y2="100"/>
    <line x1="60" y1="150" x2="200" y2="50"/><line x1="60" y1="150" x2="200" y2="150"/>
    <!-- kept only -->
    <line x1="200" y1="50"  x2="300" y2="100" stroke="#5b9bd5" stroke-width="1.5"/>
    <line x1="200" y1="100" x2="300" y2="100" stroke="#5b9bd5" stroke-width="1.5"/>
  </g>
  <!-- Input nodes -->
  <circle cx="60"  cy="50"  r="14" fill="#5b9bd5" stroke="#3a7bbf" stroke-width="1.5"/>
  <circle cx="60"  cy="100" r="14" fill="#5b9bd5" stroke="#3a7bbf" stroke-width="1.5"/>
  <circle cx="60"  cy="150" r="14" fill="#5b9bd5" stroke="#3a7bbf" stroke-width="1.5"/>
  <!-- Hidden nodes: kept -->
  <circle cx="200" cy="50"  r="14" fill="#7ecba1" stroke="#5aab81" stroke-width="1.5"/>
  <circle cx="200" cy="100" r="14" fill="#7ecba1" stroke="#5aab81" stroke-width="1.5"/>
  <!-- Hidden nodes: dropped (gray X) -->
  <circle cx="200" cy="150" r="14" fill="#e0e3ea" stroke="#aaa" stroke-width="1.5" stroke-dasharray="3,2"/>
  <line x1="192" y1="142" x2="208" y2="158" stroke="#bbb" stroke-width="2"/>
  <line x1="208" y1="142" x2="192" y2="158" stroke="#bbb" stroke-width="2"/>
  <!-- Output -->
  <circle cx="300" cy="100" r="14" fill="#e05c5c" stroke="#c03a3a" stroke-width="1.5"/>
  <text x="60"  y="54"  text-anchor="middle" fill="white" font-size="9">x₁</text>
  <text x="60"  y="104" text-anchor="middle" fill="white" font-size="9">x₂</text>
  <text x="60"  y="154" text-anchor="middle" fill="white" font-size="9">x₃</text>
  <text x="200" y="54"  text-anchor="middle" fill="white" font-size="9">a</text>
  <text x="200" y="104" text-anchor="middle" fill="white" font-size="9">a</text>
  <text x="200" y="154" text-anchor="middle" fill="#aaa"  font-size="9">✕</text>
  <text x="300" y="104" text-anchor="middle" fill="white" font-size="9">ŷ</text>
  <text x="200" y="180" text-anchor="middle" fill="#888" font-size="10">÷ p  to keep 𝔼[A] constant</text>

  <!-- Test side: all connected -->
  <g stroke="#dde0e8" stroke-width="0.9">
    <line x1="390" y1="50" x2="530" y2="50"/><line x1="390" y1="50" x2="530" y2="100"/>
    <line x1="390" y1="50" x2="530" y2="150"/>
    <line x1="390" y1="100" x2="530" y2="50"/><line x1="390" y1="100" x2="530" y2="100"/>
    <line x1="390" y1="100" x2="530" y2="150"/>
    <line x1="390" y1="150" x2="530" y2="50"/><line x1="390" y1="150" x2="530" y2="100"/>
    <line x1="390" y1="150" x2="530" y2="150"/>
    <line x1="530" y1="50" x2="630" y2="100" stroke="#5b9bd5" stroke-width="1.2"/>
    <line x1="530" y1="100" x2="630" y2="100" stroke="#5b9bd5" stroke-width="1.2"/>
    <line x1="530" y1="150" x2="630" y2="100" stroke="#5b9bd5" stroke-width="1.2"/>
  </g>
  <circle cx="390" cy="50"  r="14" fill="#5b9bd5" stroke="#3a7bbf" stroke-width="1.5"/>
  <circle cx="390" cy="100" r="14" fill="#5b9bd5" stroke="#3a7bbf" stroke-width="1.5"/>
  <circle cx="390" cy="150" r="14" fill="#5b9bd5" stroke="#3a7bbf" stroke-width="1.5"/>
  <circle cx="530" cy="50"  r="14" fill="#7ecba1" stroke="#5aab81" stroke-width="1.5"/>
  <circle cx="530" cy="100" r="14" fill="#7ecba1" stroke="#5aab81" stroke-width="1.5"/>
  <circle cx="530" cy="150" r="14" fill="#7ecba1" stroke="#5aab81" stroke-width="1.5"/>
  <circle cx="630" cy="100" r="14" fill="#e05c5c" stroke="#c03a3a" stroke-width="1.5"/>
  <text x="530" y="180" text-anchor="middle" fill="#888" font-size="10">all units active — no scaling needed</text>
</svg>

> **Why dropout works:** it forces each neuron to be useful independently — the network cannot rely on any single feature. At test time, the full network acts as an approximate average of all $2^n$ sub-networks trained during dropout.
